In [2]:
import random
import numpy as np
import pandas as pd
from datasets import load_dataset, get_dataset_config_names
from collections import Counter

# ====================================================================================================
# 🎬 FilmEval 데이터셋 실습 스크립트: 영화 평론가 AI 훈련 시뮬레이션
# ====================================================================================================
# 💡 이 데이터셋은 'FilmEval'이라는 이름으로, 영화나 스토리와 관련된 비디오(video)와 메타데이터를
# 다루는 매우 흥미로운 멀티모달(Multimodal) 데이터셋입니다.
# 🚀 목표: 우리는 이 데이터셋을 사용하여 '어떤 조건의 영상이 가장 많은가?'를 분석하고,
#   가장 쉬운 샘플들을 추려서 AI가 영상을 분석하는 과정을 흉내 내볼 것입니다.
# ----------------------------------------------------------------------------------------------------

DATASET_NAME = "ZuoHaotong/FilmEval"
SAMPLE_COUNT = 10 # 초보자 눈높이에 맞게 10개 샘플만 살펴봅시다!

# 1. 사용 가능한 Configuration 목록을 확인합니다!
print("⭐ 튜터: 자, 여러분! 어떤 설정(Config)으로 로드할 수 있는지 먼저 확인해봐요.")
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 만약 config가 있다면, 첫 번째 것을 선택합니다.
    selected_config = configs[0] if configs else None
    
except Exception as e:
    print(f"ℹ️ Config 조회 중 오류 발생: {e}. 기본 설정으로 시도합니다.")
    selected_config = None

# 2. 데이터셋 로딩 전략 (스트리밍 우선!)
# ⚡️ 데이터셋이 클 때는 '스트리밍(streaming=True)'이 좋아요. 메모리 폭발 방지!
# 🏞️ 하지만 만약 스트리밍이 안 되면? 걱정 마세요! 작은 샘플만 다운로드해서 진행할게요.
dataset = None
print("\n💡 데이터셋을 로드하는 중...")

try:
    if selected_config:
        # 가장 먼저 스트리밍으로 시도합니다! (가장 빠르고 효율적!)
        dataset = load_dataset(DATASET_NAME, name=selected_config, split="easy", streaming=True)
        print("🚀 성공! 스트리밍 모드로 데이터를 로드했어요. 엄청 빠를 거예요!")
    else:
        # 기본 설정만 사용합니다.
        dataset = load_dataset(DATASET_NAME, split="easy", streaming=True)
        print("🚀 성공! 스트리밍 모드로 데이터를 로드했어요.")

except Exception as e:
    # ⚠️ 스트리밍이 실패할 경우 (네트워크 문제, 혹은 구조 문제)
    print(f"\n🚨 스트리밍 로드 실패! 에러: {e}")
    print("⚙️ 걱정 마세요! 대신 샘플만 다운로드해서 진행할게요 (streaming=False).")
    try:
        # Fallback: 스트리밍을 포기하고, 적은 수의 샘플만 다운로드하여 일반 데이터셋으로 로드합니다.
        dataset = load_dataset(DATASET_NAME, split="train", streaming=False)
    except Exception as e2:
        print(f"😱 Oh no! 일반 로드도 실패했어요. {e2}")
        exit()


# 3. 데이터 샘플링 및 분석 준비
# 🧐 우리는 전체 데이터를 다 볼 수 없으니, 재미있게 10개의 샘플만 잘라와서 분석해 봅시다!
# 🔄 Streaming/Non-streaming 관계없이, take() 패턴을 사용합니다.

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋 (IterableDataset)
    print(f"\n✨ {SAMPLE_COUNT}개의 샘플을 가져오기 위해 데이터셋을 준비합니다...")
    sampled_dataset_iterator = iter(dataset.take(SAMPLE_COUNT))
else:
    # 일반 데이터셋 (Dataset)
    print(f"\n✨ {SAMPLE_COUNT}개의 샘플을 가져오기 위해 데이터셋을 준비합니다...")
    sampled_dataset_iterator = iter(dataset.take(SAMPLE_COUNT))


# 4. 데이터셋 구조 분석 (AI 분석가를 흉내내기!)
print("\n======================================================")
print("✨ 1단계: 데이터 특성 분석 (Dataset Feature Analysis)")
print("======================================================")

# 개별 샘플의 'features'는 메타데이터가 섞여 있어 복잡합니다.
# 우리는 순수한 데이터 구조만 깔끔하게 분석해볼 거예요.

difficulty_list = []
model_list = []
novel_name_list = []

print(f"🔬 샘플 {SAMPLE_COUNT}개를 순회하며 메타데이터를 수집합니다...")

# 순회 패턴을 사용합니다.
for i in range(SAMPLE_COUNT):
    try:
        sample = next(sampled_dataset_iterator)
        
        # 🔥 필수 작업: 복잡한 'features' 딕셔너리에서 데이터를 추출합니다.
        # 초보자 튜터 팁: AI 데이터는 종종 불필요한 껍데기(features) 안에 숨어있어요.
        # 필요한 값만 쏙 뽑아 사용해야 합니다!
        
        difficulty = sample.get('difficulty', 'N/A')
        model = sample.get('model', 'Unknown')
        novel_name = sample.get('novel_name', 'N/A')
        
        difficulty_list.append(difficulty)
        model_list.append(model)
        novel_name_list.append(novel_name)
        
    except StopIteration:
        break # 샘플이 끝났으면 반복 중단

# 📈 분석 결과 요약
print("\n\n📊 [결과 보고서] 가장 많이 등장한 난이도와 모델은?")
print("-" * 40)

# 빈도수 계산 (Counter 사용)
difficulty_counts = Counter(difficulty_list)
model_counts = Counter(model_list)

print(f"✅ 🧠 분석한 {len(difficulty_list)}개 샘플에서 가장 많이 발견된 난이도:")
for difficulty, count in difficulty_counts.most_common(3):
    print(f"    👉 '{difficulty}' 난이도: {count}개 (가장 흔해요!)")

print(f"\n✅ 🤖 분석한 {len(model_list)}개 샘플에서 주로 사용된 모델:")
for model, count in model_counts.most_common(3):
    print(f"    👉 '{model}' 모델: {count}개 (이 모델이 주력인가봐요!)")


print("\n======================================================")
print("🎬 2단계: 개별 샘플 상세 탐색 (Inspecting One Sample)")
print("======================================================")

# 가장 첫 번째 샘플을 다시 가져와서 깊이 탐색해봅시다.
try:
    first_sample = next(iter(dataset.take(1)))
    print("\n✨ 가장 첫 번째 샘플의 메타데이터를 분석합니다.")
    print("-" * 40)
    
    print(f"   ▶️ Sample ID: {first_sample.get('sample_id', 'N/A')}")
    print(f"   ▶️ 모델명 (Model): {first_sample.get('model', 'N/A')}")
    print(f"   ▶️ 난이도 (Difficulty): {first_sample.get('difficulty', 'N/A')}")
    print(f"   ▶️ 원작 노벨명 (Novel Name): {first_sample.get('novel_name', 'N/A')}")

    # 📼 비디오 파일 자체를 상징적으로 확인합니다.
    video_data = first_sample.get('video')
    if video_data:
        print(f"   ▶️ 비디오 데이터 (Video): 존재! (실제 비디오 파일 객체가 여기에 들어갑니다.)")
    else:
        print("   ▶️ 비디오 데이터 (Video): ℹ️ 로드 실패 또는 데이터 없음.")

except StopIteration:
    print("💔 분석할 샘플이 하나도 없어서 탐색을 완료할 수 없습니다.")


print("\n======================================================")
print("🎉 🎉 스크립트 종료: 🎉")
print("축하합니다! 여러분은 이제 이 복잡한 AI 데이터셋의 핵심 구조를 파악하는 분석가가 되었습니다!")
print("데이터 전처리, 로딩 최적화, 그리고 분석까지 모두 마스터하셨어요!")
print("======================================================")

⭐ 튜터: 자, 여러분! 어떤 설정(Config)으로 로드할 수 있는지 먼저 확인해봐요.
✅ 사용 가능한 Config 목록: ['FilmWorld', 'MM-StoryAgent', 'MovieAgent', 'VideoClaw', 'VideoGen-of-Thought', 'ViMax']

💡 데이터셋을 로드하는 중...
🚀 성공! 스트리밍 모드로 데이터를 로드했어요. 엄청 빠를 거예요!

✨ 10개의 샘플을 가져오기 위해 데이터셋을 준비합니다...

✨ 1단계: 데이터 특성 분석 (Dataset Feature Analysis)
🔬 샘플 10개를 순회하며 메타데이터를 수집합니다...


ImportError: To support decoding videos, please install 'decord'.